# SageMaker SGLang endpoint example (Simplified Version)

This notebook deploys SGLang using its **native SageMaker API support**. 

**Key Features:**
- ✅ SGLang natively supports `/ping` and `/invocations` endpoints
- ✅ No proxy layer needed - direct deployment
- ✅ Simpler architecture and faster startup
- ⚠️ Only supports **Chat Completions API** (messages format)

**Note:** If you need both Chat and Completion APIs, use the proxy-based version.

## 1. Define some variables

The byoc will build and store a SGLang endpoint docker image in you ECR private repo (for example `sagemaker_endpoint/sglang`), you need to define the following variables.

In [1]:
MODEL_ID = "Qwen/Qwen3.5-0.8B"
# INSTANCE_TYPE = "ml.p4d.24xlarge"
INSTANCE_TYPE = "ml.g6.2xlarge"
SGLANG_REPO = "lmsysorg/sglang"
SGLANG_VERSION = "v0.5.9"
REPO_NAMESPACE = "sagemaker_endpoint/sglang"
ACCOUNT = !aws sts get-caller-identity --query Account --output text
REGION = !aws configure get region
ACCOUNT = ACCOUNT[0]
REGION = REGION[0]
# If it can't access SGLang docker repo in China Region, you need to handle it by yourself.
# if REGION.startswith("cn"):
#     # this is a example repo port from lmsysorg/sglang, you can create your own docker image in your global region account
#     SGLANG_REPO = "public.ecr.aws/your-repo/sglang"
#     CONTAINER = f"{ACCOUNT}.dkr.ecr.{REGION}.amazonaws.com.cn/{REPO_NAMESPACE}:{SGLANG_VERSION}"
# else:
CONTAINER = f"{ACCOUNT}.dkr.ecr.{REGION}.amazonaws.com/{REPO_NAMESPACE}:{SGLANG_VERSION}"

## 2. Build the container

Endpoint starting codes are in `app/`. The script will build and push to ecr. 

**The docker only need to be built once**, and after that, when deploying other endpoints, the same docker image can be shared.

In [2]:
cmd = f"SGLANG_REPO={SGLANG_REPO} SGLANG_VERSION={SGLANG_VERSION} REPO_NAMESPACE={REPO_NAMESPACE} ACCOUNT={ACCOUNT} REGION={REGION} bash ./build_and_push.sh"
print("Running:", cmd)
!{cmd}

Running: SGLANG_REPO=lmsysorg/sglang SGLANG_VERSION=v0.5.9 REPO_NAMESPACE=sagemaker_endpoint/sglang ACCOUNT=152955032929 REGION=us-east-1 bash ./build_and_push.sh
WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store

Login Succeeded
152955032929.dkr.ecr.us-east-1.amazonaws.com/sagemaker_endpoint/sglang:v0.5.9
#0 building with "default" instance using docker driver

#1 [internal] load build definition from dockerfile
#1 transferring dockerfile: 716B done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/lmsysorg/sglang:v0.5.9
#2 DONE 0.1s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [1/4] FROM docker.io/lmsysorg/sglang:v0.5.9@sha256:e216b7dc4ac1938b599b982233ccf7eb2b11dd1f07fc2e00a7b9841052c553be
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 1.09kB

## 3. Deploy on SageMaker

define the model and deploy on SageMaker

In [54]:
%pip install -U "boto3==1.42.68" "sagemaker<3" transformers huggingface_hub modelscope

Note: you may need to restart the kernel to use updated packages.


### 3.1 Init SageMaker session

In [3]:
import os
import re
import json
from datetime import datetime
import time

import boto3
import sagemaker

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
default_bucket = sess.default_bucket()

sagemaker_client = boto3.client("sagemaker")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### 3.2 Download and upload model file

Firstly, you need to prepare model weights and upload to S3. You can download from HuggingFace, ModelScope or upload your own model. 

If you want SGLang to automatically pull the model when it starts, this step can be skipped.

In [4]:
model_name = MODEL_ID.replace("/", "-").replace(".", "-")
local_model_path = os.environ['HOME'] + "/models/" + model_name
s3_model_path = f"s3://{default_bucket}/models/" + model_name

%mkdir -p code {local_model_path}

print("local_model_path:", local_model_path)

local_model_path: /home/ec2-user/models/Qwen-Qwen3-5-0-8B


##### Option 1: Global region (download from HuggingFace)

In [5]:
from huggingface_hub import snapshot_download
# 方法2: 带更多配置选项
model_path = snapshot_download(
    repo_id=f"{MODEL_ID}",
    local_dir=f"{local_model_path}",
    resume_download=True,
    max_workers=8,
    # token="YOUR_HF_TOKEN",  # 如果是私有模型或需要认证
    local_dir_use_symlinks=False,  # 不使用符号链接，直接复制文件
    ignore_patterns=["*.msgpack", "*.h5"],  # 忽略某些文件
    allow_patterns=["*.safetensors", "*.json", "*.txt"],  # 只下载特定文件
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

##### Option 2: China region  (download from ModelScope)

In [ ]:
!pip install modelscope

In [ ]:
!modelscope download --local_dir {local_model_path} {MODEL_ID} 

#### upload to s3

In [6]:
!aws s3 sync {local_model_path} {s3_model_path}
print("s3_model_path:", s3_model_path)

upload: ../../../../models/Qwen-Qwen3-5-0-8B/.cache/huggingface/download/config.json.lock to s3://sagemaker-us-east-1-152955032929/models/Qwen-Qwen3-5-0-8B/.cache/huggingface/download/config.json.lock
upload: ../../../../models/Qwen-Qwen3-5-0-8B/.cache/huggingface/download/model.safetensors-00001-of-00001.safetensors.lock to s3://sagemaker-us-east-1-152955032929/models/Qwen-Qwen3-5-0-8B/.cache/huggingface/download/model.safetensors-00001-of-00001.safetensors.lock
upload: ../../../../models/Qwen-Qwen3-5-0-8B/.cache/huggingface/download/merges.txt.lock to s3://sagemaker-us-east-1-152955032929/models/Qwen-Qwen3-5-0-8B/.cache/huggingface/download/merges.txt.lock
upload: ../../../../models/Qwen-Qwen3-5-0-8B/.cache/huggingface/download/tokenizer.json.lock to s3://sagemaker-us-east-1-152955032929/models/Qwen-Qwen3-5-0-8B/.cache/huggingface/download/tokenizer.json.lock
upload: ../../../../models/Qwen-Qwen3-5-0-8B/.cache/huggingface/download/tokenizer_config.json.lock to s3://sagemaker-us-east-

### 3.3 Prepare SGLang start scripts

Then you need to write the SGLang starting scripts for endpoint, the container will automatically use the `start.sh` as the entrypoint.

Please carefully modify the startup script file as needed, such as the model running parameter information. All parameters can be referenced at [https://sglang.readthedocs.io/](https://sglang.readthedocs.io/)

Here is a simple script that pulling a model from S3 and starting a SGLang server.

In [7]:
endpoint_model_name = sagemaker.utils.name_from_base(model_name, short=True)
local_code_path = endpoint_model_name
s3_code_path = f"s3://{default_bucket}/endpoint_code/sglang_byoc/{endpoint_model_name}.tar.gz"

%mkdir -p {local_code_path}

print("local_code_path:", local_code_path)

with open(f"{local_code_path}/start.sh", "w") as f:
    f.write(f"""#!/bin/bash

# download model to local
s5cmd sync {s3_model_path}/* /opt/ml/modelfile/

# the start script need to be adjust as you needed
# port needs to be $SAGEMAKER_BIND_TO_PORT
# SGLang parameter reference: https://sglang.readthedocs.io/en/latest/references/sampling_params.html

python3 -m sglang.launch_server \\
    --host 0.0.0.0 \\
    --port 8080 \\
    --model-path /opt/ml/modelfile/ \\
    --trust-remote-code \\
    --context-length 32768 \\
    --mem-fraction-static 0.85
""")

local_code_path: Qwen-Qwen3-5-0-8B-260315-1216


In [8]:
!rm -f {local_code_path}.tar.gz
!tar czvf {local_code_path}.tar.gz {local_code_path}/
!aws s3 cp {local_code_path}.tar.gz {s3_code_path}
print("s3_code_path:", s3_code_path)

Qwen-Qwen3-5-0-8B-260315-1216/
Qwen-Qwen3-5-0-8B-260315-1216/start.sh
upload: ./Qwen-Qwen3-5-0-8B-260315-1216.tar.gz to s3://sagemaker-us-east-1-152955032929/endpoint_code/sglang_byoc/Qwen-Qwen3-5-0-8B-260315-1216.tar.gz
s3_code_path: s3://sagemaker-us-east-1-152955032929/endpoint_code/sglang_byoc/Qwen-Qwen3-5-0-8B-260315-1216.tar.gz


### 3.3 Deploy endpoint on SageMaker

In [9]:
# Step 0. create model
subnets = "" # subnet-0e032296595b95caa
sg_ids = ""
vpc_config = None
if subnets and sg_ids:
    vpc_config = {
        "Subnets": subnets.split(","),
        "SecurityGroupIds": sg_ids.split(",")
    }

create_model_params = {
    "ModelName": endpoint_model_name,
    "PrimaryContainer": {
        "Image": CONTAINER,
        "ModelDataUrl": s3_code_path,
    },
    "ExecutionRoleArn": role,
}
if vpc_config:
    create_model_params["VpcConfig"] = vpc_config

create_model_response = sagemaker_client.create_model(**create_model_params)

# endpoint_model_name already defined in above step

print(create_model_response)
print("endpoint_model_name:", endpoint_model_name)

{'ModelArn': 'arn:aws:sagemaker:us-east-1:152955032929:model/Qwen-Qwen3-5-0-8B-260315-1216', 'ResponseMetadata': {'RequestId': '86e2babc-86b5-47c5-9d81-5d3f40d79439', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '86e2babc-86b5-47c5-9d81-5d3f40d79439', 'strict-transport-security': 'max-age=47304000; includeSubDomains', 'x-frame-options': 'DENY', 'content-security-policy': "frame-ancestors 'none'", 'cache-control': 'no-cache, no-store, must-revalidate', 'x-content-type-options': 'nosniff', 'content-type': 'application/x-amz-json-1.1', 'content-length': '91', 'date': 'Sun, 15 Mar 2026 12:16:24 GMT'}, 'RetryAttempts': 0}}
endpoint_model_name: Qwen-Qwen3-5-0-8B-260315-1216


In [10]:
# Step 1. create endpoint config

endpoint_config_name = sagemaker.utils.name_from_base(model_name, short=True)

production_variant = {
    "VariantName": "AllTraffic",
    "ModelName": endpoint_model_name,
    "InitialInstanceCount": 1,
    "InstanceType": INSTANCE_TYPE,
    "InitialVariantWeight": 1.0,
    "ContainerStartupHealthCheckTimeoutInSeconds": 1000,
    "InferenceAmiVersion": "al2-ami-sagemaker-inference-gpu-3-1"
}

# fill with FTP ARN
# "MlReservationArn": "arn:aws:sagemaker:region:account:ml-reservation/my-capacity"
# ml_reservation_arn = "arn:aws:sagemaker:us-east-1:152955032929:training-plan/shein-byoc-test-2"
ml_reservation_arn = None
# ml_reservation_arn = "arn:aws:sagemaker:us-east-1:152955032929:reserved-capacity/d1rvcsqywchy7sa453c5oba9l"
if ml_reservation_arn:
    production_variant["CapacityReservationConfig"] = {
        "MlReservationArn": ml_reservation_arn,
        "CapacityReservationPreference": "capacity-reservations-only"
    }

create_config_params = {
    "EndpointConfigName": endpoint_config_name,
    "ProductionVariants": [production_variant],
}

endpoint_config_response = sagemaker_client.create_endpoint_config(**create_config_params)

print(endpoint_config_response)
print("endpoint_config_name:", endpoint_config_name)

{'EndpointConfigArn': 'arn:aws:sagemaker:us-east-1:152955032929:endpoint-config/Qwen-Qwen3-5-0-8B-260315-1216', 'ResponseMetadata': {'RequestId': '7d849981-8c4a-4377-ac22-9e3bfa91e448', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '7d849981-8c4a-4377-ac22-9e3bfa91e448', 'strict-transport-security': 'max-age=47304000; includeSubDomains', 'x-frame-options': 'DENY', 'content-security-policy': "frame-ancestors 'none'", 'cache-control': 'no-cache, no-store, must-revalidate', 'x-content-type-options': 'nosniff', 'content-type': 'application/x-amz-json-1.1', 'content-length': '110', 'date': 'Sun, 15 Mar 2026 12:16:27 GMT'}, 'RetryAttempts': 0}}
endpoint_config_name: Qwen-Qwen3-5-0-8B-260315-1216


In [11]:
# Step 2. create endpoint

endpoint_name = sagemaker.utils.name_from_base(model_name, short=True)

create_endpoint_response = sagemaker_client.create_endpoint(
    EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name
)
print(create_endpoint_response)
print("endpoint_config_name:", endpoint_name)
while 1:
    status = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
    if status != "Creating":
        break
    print(datetime.now().strftime('%Y%m%d-%H:%M:%S') + " status: " + status)
    time.sleep(60)
print("Endpoint created:", endpoint_name)

{'EndpointArn': 'arn:aws:sagemaker:us-east-1:152955032929:endpoint/Qwen-Qwen3-5-0-8B-260315-1216', 'ResponseMetadata': {'RequestId': '49c114a9-cd50-4be3-8fd0-0ab38a17c879', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '49c114a9-cd50-4be3-8fd0-0ab38a17c879', 'strict-transport-security': 'max-age=47304000; includeSubDomains', 'x-frame-options': 'DENY', 'content-security-policy': "frame-ancestors 'none'", 'cache-control': 'no-cache, no-store, must-revalidate', 'x-content-type-options': 'nosniff', 'content-type': 'application/x-amz-json-1.1', 'content-length': '97', 'date': 'Sun, 15 Mar 2026 12:16:32 GMT'}, 'RetryAttempts': 0}}
endpoint_config_name: Qwen-Qwen3-5-0-8B-260315-1216
20260315-12:16:32 status: Creating
20260315-12:17:33 status: Creating
20260315-12:18:33 status: Creating
20260315-12:19:33 status: Creating
20260315-12:20:33 status: Creating
20260315-12:21:33 status: Creating
20260315-12:22:33 status: Creating
20260315-12:23:33 status: Creating
Endpoint created: Qwen

## 4. Test

You can invoke your model with SageMaker runtime.

In [12]:
messages = [{
        "role": "user",
        "content": "Write a quick sort in python"
}]

### 4.1 Message api non-stream mode

In [ ]:
sagemaker_runtime = boto3.client('runtime.sagemaker')

payload = {
    "messages": messages,
    "max_tokens": 1024,
    "stream": False
}
response = sagemaker_runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType='application/json',
    Body=json.dumps(payload)
)

print(json.loads(response['Body'].read())["choices"][0]["message"]["content"])

### 4.2 Message api stream mode

In [ ]:
payload = {
    "messages": messages,
    "max_tokens": 1024,
    "stream": True
}

response = sagemaker_runtime.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    ContentType='application/json',
    Body=json.dumps(payload)
)

buffer = ""
for t in response['Body']:
    buffer += t["PayloadPart"]["Bytes"].decode()
    last_idx = 0
    for match in re.finditer(r'^data:\s*(.+?)(\n\n)', buffer):
        try:
            data = json.loads(match.group(1).strip())
            last_idx = match.span()[1]
            print(data["choices"][0]["delta"]["content"], end="")
        except (json.JSONDecodeError, KeyError, IndexError) as e:
            pass
    buffer = buffer[last_idx:]
print()

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(local_model_path, trust_remote_code=True)

prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

payload = {
    "prompt": prompt,
    "max_tokens": 1024,
    "stream": False
}

response = sagemaker_runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType='application/json',
    Body=json.dumps(payload)
)

print(json.loads(response['Body'].read())["choices"][0]["text"])

In [ ]:
payload = {
    "prompt": prompt,
    "max_tokens": 1024,
    "stream": True
}

response = sagemaker_runtime.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    ContentType='application/json',
    Body=json.dumps(payload)
)

buffer = ""
for t in response['Body']:
    buffer += t["PayloadPart"]["Bytes"].decode()
    last_idx = 0
    for match in re.finditer(r'^data:\s*(.+?)(\n\n)', buffer):
        try:
            data = json.loads(match.group(1).strip())
            last_idx = match.end()
            # print(data)
            print(data["choices"][0]["text"], end="")
        except (json.JSONDecodeError, KeyError, IndexError) as e:
            pass
    buffer = buffer[last_idx:]
print()